# TimesFM 2.5 zero-shot experiment — Walmart weekly sales

This first foundation-model experiment performs **no training or fine-tuning**. It downloads Google's pretrained TimesFM 2.5 200M checkpoint, gives it each Store–Dept sales history, and forecasts the same final 39-week validation horizon used elsewhere in the project.

The notebook evaluates original-scale WMAE, compares TimesFM with a 52-week seasonal-naive forecast, logs diagnostics and artifacts to W&B, and is designed for a Colab T4 GPU.

In [ ]:
%pip install -q -U "transformers>=5.3.0" accelerate wandb

In [ ]:
import gc
import hashlib
import json
import os
import platform
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
import wandb
from transformers import TimesFm2_5ModelForPrediction

warnings.filterwarnings('ignore')
torch.set_float32_matmul_precision('high')
print({'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})

In [ ]:
# Colab: change only DATA_DIR if your files are stored elsewhere. train.csv.zip is supported.
CONFIG = {
    'data_dir': '/content/drive/MyDrive/walmart_competition_data',
    'output_dir': '/content/artifacts/timesfm_zero_shot_v1',
    'model_id': 'google/timesfm-2.5-200m-transformers',
    'validation_weeks': 39,
    'seasonal_period': 52,
    'batch_size': 64,
    'min_context_points': 32,
    'holiday_weight': 5.0,
    'clip_min': 0.0,
    'clip_max': 300000.0,
    'wandb_entity': 'kende23-n-a',
    'wandb_project': 'Walmart-Recruiting---Store-Sales-Forecasting',
    'wandb_run_name': 'timesfm_v1_zero_shot_all_series_39w',
    'seed': 42,
}
DATA_DIR = Path(CONFIG['data_dir'])
OUTPUT_DIR = Path(CONFIG['output_dir'])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(CONFIG['seed'])
print(CONFIG)

In [ ]:
# Authenticate W&B from Colab Secrets when available; otherwise wandb.login() opens the normal prompt.
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
wandb.login(key=wandb_key) if wandb_key else wandb.login()

In [ ]:
def locate_csv(data_dir: Path, filename: str) -> Path:
    plain = data_dir / filename
    zipped = data_dir / f'{filename}.zip'
    if plain.exists():
        return plain
    if zipped.exists():
        return zipped
    raise FileNotFoundError(f'Missing {plain} and {zipped}')

train_path = locate_csv(DATA_DIR, 'train.csv')
train_raw = pd.read_csv(train_path, parse_dates=['Date'])
train_raw['Store'] = train_raw['Store'].astype('int16')
train_raw['Dept'] = train_raw['Dept'].astype('int16')
train_raw['Weekly_Sales'] = pd.to_numeric(train_raw['Weekly_Sales'], errors='coerce').fillna(0.0).astype('float32')
train_raw['IsHoliday'] = train_raw['IsHoliday'].astype(bool)
train_raw = train_raw.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
assert not train_raw.duplicated(['Store', 'Dept', 'Date']).any()
print({'train_path': str(train_path), 'rows': len(train_raw), 'series': train_raw.groupby(['Store', 'Dept']).ngroups, 'date_min': str(train_raw.Date.min().date()), 'date_max': str(train_raw.Date.max().date())})

## Honest final-39-week validation preparation

TimesFM requires an evenly spaced sequence. Each Store–Dept history is therefore reindexed to the global Friday calendar and absent observations are represented as zero. Metrics are calculated only on rows that actually exist in the original validation data.

In [ ]:
all_dates = pd.DatetimeIndex(sorted(train_raw['Date'].unique()))
validation_dates = all_dates[-CONFIG['validation_weeks']:]
history_dates = all_dates[:-CONFIG['validation_weeks']]
split_date = validation_dates[0]

validation_raw = train_raw[train_raw['Date'].isin(validation_dates)].copy()
validation_keys = validation_raw[['Store', 'Dept']].drop_duplicates().sort_values(['Store', 'Dept'])
series_keys = [tuple(x) for x in validation_keys.to_numpy()]

sales_lookup = train_raw.set_index(['Store', 'Dept', 'Date'])['Weekly_Sales']
holiday_by_date = train_raw.groupby('Date')['IsHoliday'].max().reindex(validation_dates).fillna(False).astype(bool)
print({'history_weeks': len(history_dates), 'validation_weeks': len(validation_dates), 'validation_start': str(validation_dates.min().date()), 'validation_end': str(validation_dates.max().date()), 'validation_rows': len(validation_raw), 'validation_series': len(series_keys)})

In [ ]:
def wmae(actual, predicted, is_holiday, holiday_weight=5.0):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.sum(weights * np.abs(actual - predicted)) / np.sum(weights))

contexts, seasonal_forecasts, usable_keys, fallback_keys = [], [], [], []
for store, dept in series_keys:
    full = sales_lookup.reindex(pd.MultiIndex.from_product([[store], [dept], all_dates], names=['Store', 'Dept', 'Date'])).fillna(0.0).to_numpy(dtype=np.float32)
    context = full[:-CONFIG['validation_weeks']]
    seasonal = full[-CONFIG['validation_weeks'] - CONFIG['seasonal_period']:-CONFIG['seasonal_period']]
    if len(seasonal) != CONFIG['validation_weeks']:
        seasonal = np.resize(seasonal, CONFIG['validation_weeks']).astype(np.float32) if len(seasonal) else np.zeros(CONFIG['validation_weeks'], dtype=np.float32)
    seasonal_forecasts.append(np.clip(seasonal, CONFIG['clip_min'], CONFIG['clip_max']))
    if len(context) >= CONFIG['min_context_points'] and np.isfinite(context).all():
        contexts.append(context)
        usable_keys.append((store, dept))
    else:
        fallback_keys.append((store, dept))

seasonal_by_key = dict(zip(series_keys, seasonal_forecasts))
print({'timesfm_series': len(usable_keys), 'fallback_series': len(fallback_keys), 'context_length': len(history_dates)})

## Load pretrained TimesFM 2.5 and run zero-shot forecasting

There is no optimizer, epoch, backward pass, or target-specific fitting in this experiment. The checkpoint is used exactly as pretrained.

In [ ]:
if not torch.cuda.is_available():
    print('WARNING: GPU is not enabled. In Colab select Runtime > Change runtime type > T4 GPU.')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
load_started = time.time()
model = TimesFm2_5ModelForPrediction.from_pretrained(
    CONFIG['model_id'],
    torch_dtype=torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
)
model.eval()
load_seconds = time.time() - load_started
parameter_count = sum(p.numel() for p in model.parameters())
print({'device': str(model.device), 'parameters': parameter_count, 'load_seconds': load_seconds})

In [ ]:
forecast_started = time.time()
timesfm_by_key = {}
batch_size = CONFIG['batch_size']

with torch.inference_mode():
    for start in range(0, len(contexts), batch_size):
        batch_np = contexts[start:start + batch_size]
        batch_keys = usable_keys[start:start + batch_size]
        batch_tensors = [torch.as_tensor(x, dtype=torch.float32, device=model.device) for x in batch_np]
        outputs = model(past_values=batch_tensors, return_dict=True)
        batch_forecasts = outputs.mean_predictions[:, :CONFIG['validation_weeks']].detach().float().cpu().numpy()
        batch_forecasts = np.nan_to_num(batch_forecasts, nan=0.0, posinf=CONFIG['clip_max'], neginf=0.0)
        batch_forecasts = np.clip(batch_forecasts, CONFIG['clip_min'], CONFIG['clip_max'])
        for key, forecast in zip(batch_keys, batch_forecasts):
            timesfm_by_key[key] = forecast.astype(np.float32)
        if start == 0 or (start // batch_size + 1) % 10 == 0:
            print({'finished': min(start + batch_size, len(contexts)), 'total': len(contexts), 'elapsed_min': round((time.time() - forecast_started) / 60, 2)})

for key in fallback_keys:
    timesfm_by_key[key] = seasonal_by_key[key].copy()
forecast_seconds = time.time() - forecast_started
if torch.cuda.is_available():
    peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_gpu_gb = 0.0
print({'forecast_minutes': forecast_seconds / 60, 'peak_gpu_gb': peak_gpu_gb, 'predicted_series': len(timesfm_by_key)})

## Evaluate original-scale validation WMAE

In [ ]:
date_to_horizon = {date: i for i, date in enumerate(validation_dates)}
records = []
for row in validation_raw.itertuples(index=False):
    key = (row.Store, row.Dept)
    h = date_to_horizon[row.Date]
    records.append({
        'Store': int(row.Store), 'Dept': int(row.Dept), 'Date': row.Date,
        'IsHoliday': bool(row.IsHoliday), 'Weekly_Sales': float(row.Weekly_Sales),
        'TimesFM_Prediction': float(timesfm_by_key[key][h]),
        'SeasonalNaive52_Prediction': float(seasonal_by_key[key][h]),
        'Horizon': h + 1,
        'UsedFallback': key in set(fallback_keys),
    })
validation_predictions = pd.DataFrame(records).sort_values(['Date', 'Store', 'Dept']).reset_index(drop=True)
timesfm_wmae = wmae(validation_predictions['Weekly_Sales'], validation_predictions['TimesFM_Prediction'], validation_predictions['IsHoliday'], CONFIG['holiday_weight'])
seasonal_wmae = wmae(validation_predictions['Weekly_Sales'], validation_predictions['SeasonalNaive52_Prediction'], validation_predictions['IsHoliday'], CONFIG['holiday_weight'])
timesfm_mae = float(np.mean(np.abs(validation_predictions['Weekly_Sales'] - validation_predictions['TimesFM_Prediction'])))
improvement_pct = 100.0 * (seasonal_wmae - timesfm_wmae) / seasonal_wmae
prediction_hash = hashlib.sha256(validation_predictions['TimesFM_Prediction'].to_numpy(dtype=np.float64).tobytes()).hexdigest()
metrics = {
    'validation/wmae': timesfm_wmae,
    'validation/mae': timesfm_mae,
    'validation/seasonal_naive_wmae': seasonal_wmae,
    'validation/improvement_vs_seasonal_naive_pct': improvement_pct,
    'coverage/total_series': len(series_keys),
    'coverage/timesfm_series': len(usable_keys),
    'coverage/fallback_series': len(fallback_keys),
    'runtime/model_load_seconds': load_seconds,
    'runtime/forecast_minutes': forecast_seconds / 60,
    'runtime/peak_gpu_gb': peak_gpu_gb,
}
print(metrics)
print({'prediction_sha256': prediction_hash})
display(validation_predictions.head())

In [ ]:
weekly_diagnostics = validation_predictions.groupby(['Date', 'IsHoliday'], as_index=False).apply(
    lambda g: pd.Series({
        'TimesFM_MAE': np.mean(np.abs(g['Weekly_Sales'] - g['TimesFM_Prediction'])),
        'SeasonalNaive_MAE': np.mean(np.abs(g['Weekly_Sales'] - g['SeasonalNaive52_Prediction'])),
        'Rows': len(g),
    }), include_groups=False
).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(weekly_diagnostics['Date'], weekly_diagnostics['TimesFM_MAE'], label='TimesFM')
axes[0].plot(weekly_diagnostics['Date'], weekly_diagnostics['SeasonalNaive_MAE'], label='Seasonal naive', alpha=0.8)
axes[0].set_title('Weekly validation MAE'); axes[0].tick_params(axis='x', rotation=45); axes[0].legend()
sample = validation_predictions.sample(min(12000, len(validation_predictions)), random_state=CONFIG['seed'])
axes[1].scatter(sample['Weekly_Sales'], sample['TimesFM_Prediction'], s=6, alpha=0.25)
limit = np.percentile(np.r_[sample['Weekly_Sales'], sample['TimesFM_Prediction']], 99)
axes[1].plot([0, limit], [0, limit], '--', color='black'); axes[1].set_xlim(0, limit); axes[1].set_ylim(0, limit)
axes[1].set_title('Actual vs TimesFM prediction'); axes[1].set_xlabel('Actual'); axes[1].set_ylabel('Prediction')
plt.tight_layout()
diagnostic_path = OUTPUT_DIR / 'timesfm_zero_shot_diagnostics.png'
fig.savefig(diagnostic_path, dpi=160, bbox_inches='tight')
plt.show()

## Log the complete zero-shot experiment to W&B

This is an evaluation artifact, not a trained-model artifact. The immutable pretrained checkpoint is identified by its Hugging Face model ID; a raw-input pipeline and Model Registry entry will only be created after a TimesFM configuration wins validation.

In [ ]:
predictions_path = OUTPUT_DIR / 'timesfm_zero_shot_validation_predictions.csv'
weekly_path = OUTPUT_DIR / 'timesfm_zero_shot_weekly_diagnostics.csv'
metrics_path = OUTPUT_DIR / 'timesfm_zero_shot_metrics.json'
validation_predictions.to_csv(predictions_path, index=False)
weekly_diagnostics.to_csv(weekly_path, index=False)
manifest = {
    'experiment': 'TimesFM 2.5 zero-shot',
    'model_id': CONFIG['model_id'],
    'fine_tuned': False,
    'validation_start': str(validation_dates.min().date()),
    'validation_end': str(validation_dates.max().date()),
    'prediction_sha256': prediction_hash,
    **metrics,
}
metrics_path.write_text(json.dumps(manifest, indent=2))

run = wandb.init(
    entity=CONFIG['wandb_entity'], project=CONFIG['wandb_project'],
    group='timesfm-experiments', job_type='zero_shot_validation',
    name=CONFIG['wandb_run_name'], config=CONFIG,
)
run.log(metrics)
run.log({
    'validation/prediction_table': wandb.Table(dataframe=validation_predictions.head(20000)),
    'validation/weekly_diagnostics': wandb.Table(dataframe=weekly_diagnostics),
    'validation/diagnostic_plot': wandb.Image(str(diagnostic_path)),
})
artifact = wandb.Artifact(
    name='timesfm-v1-zero-shot-validation', type='evaluation',
    description='TimesFM 2.5 200M zero-shot predictions on the final 39 Walmart validation weeks.',
    metadata=manifest,
)
for path in [predictions_path, weekly_path, metrics_path, diagnostic_path]:
    artifact.add_file(str(path))
run.log_artifact(artifact, aliases=['v1', 'latest'])
run.summary.update(manifest)
run.finish()
print({'wandb_logged': True, 'artifact': artifact.name, 'output_dir': str(OUTPUT_DIR)})

## Reading the result

- `validation/wmae` is the TimesFM zero-shot score; lower is better.
- `validation/seasonal_naive_wmae` is the same-row 52-week reference.
- Positive `improvement_vs_seasonal_naive_pct` means TimesFM won.
- This notebook intentionally does not tune, blend, fine-tune, register a pipeline, or generate a Kaggle submission. Those decisions should follow the measured zero-shot result.